# RAG Chatbot with BioBERT and LLaMA 2

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline using biomedical embeddings (`BioBERT`) and a local language model (`LLaMA 2`) to answer domain-specific medical questions. The pipeline includes:

- Document loading and splitting  
- Embedding generation with `BioBERT`  
- Vector store creation using `Chroma`  
- Retrieval-based question answering with `LLaMA 2`  

This setup is ideal for medical QA tasks where factual grounding and domain relevance are critical.  
The model runs locally via `transformers` and Hugging Face pipelines, without requiring an API key.

---

In [16]:
!pip install -U langchain langchain-community langchain-huggingface
!pip install -U chromadb
!pip install -U sentence-transformers
!pip install -U transformers
!pip install -U accelerate
!pip install -U langchain-chroma

In [18]:
import os
import glob
import shutil

from huggingface_hub import login

from langchain.schema import Document
from langchain.document_loaders import WikipediaLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

### Project Directory Connection


In [4]:
!git clone https://github.com/a20190202/PLN_Medical_Flashcard.git

Cloning into 'PLN_Medical_Flashcard'...
remote: Enumerating objects: 280, done.
remote: Counting objects: 100% (280/280), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 280 (delta 94), reused 247 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (280/280), 38.27 MiB | 8.42 MiB/s, done.
Resolving deltas: 100% (94/94), done.
Updating files: 100% (76/76), done.


In [5]:
!ls

PLN_Medical_Flashcard  sample_data


In [6]:
%cd PLN_Medical_Flashcard/pln_model

/content/PLN_Medical_Flashcard/pln_model


In [7]:
!pwd

/content/PLN_Medical_Flashcard/pln_model


# Vectorstore Generation
---

In [8]:
def read_txt_files(folder_path):
    all_docs = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        length_function=len,
    )

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

            chunks = splitter.split_text(text)

            for i, chunk in enumerate(chunks):
                doc = Document(
                    page_content=chunk,
                    metadata={
                        "source": filename,
                        "chunk_id": i,
                        "total_chunks": len(chunks)
                    }
                )
                all_docs.append(doc)

    return all_docs

all_documents = read_txt_files("data/textbooks")

## Embeddings model
### `pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb`

This SentenceTransformer model is a fine-tuned version of BioBERT on multiple natural language inference (NLI) and semantic similarity datasets, including:

- MNLI, SNLI, SciNLI, SciTail, MedNLI, and STS-B

It is specifically optimized for semantic similarity tasks in the biomedical domain and is suitable for generating high-quality dense vector embeddings of medical questions, terms, or documents.

- **Base model:** `dmis-lab/biobert-base-cased-v1.1`  
- **Embedding dimensions:** `768`  
- **Use case:** Biomedical sentence embeddings for retrieval and clustering

Model link: [https://huggingface.co/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb](https://huggingface.co/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb)


In [9]:
EMBEDDINGS = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"

In [10]:
embeddings_model = HuggingFaceEmbeddings(
        model_name=EMBEDDINGS,
    )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
# Create Vector Store (Run Only Once)
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings_model,
    persist_directory=f"./{EMBEDDINGS.replace('/','_')}"
)

### Save the vector store after creation

In [14]:
# Define original path (correct one where Chroma actually saved the files)
vectorstore_dir = f"./{EMBEDDINGS.replace('/', '_')}"
zip_path = f"{vectorstore_dir}.zip"

# Create the ZIP file from the original directory
shutil.make_archive(vectorstore_dir, 'zip', vectorstore_dir)

# Move the ZIP to /content so it's visible in Colab file browser
!mv "{zip_path}" /content/

print(f"Vector store zipped and moved to /content/: {os.path.basename(zip_path)}")

✅ Vector store zipped and moved to /content/: pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb.zip


### Load the saved vector store

In [19]:
# Unzip the saved vector store
vectorstore_dir = f"./{EMBEDDINGS.replace('/', '_')}"
zip_path = f"/content/{EMBEDDINGS.replace('/', '_')}.zip"

# Unzip only if not already extracted
if not os.path.exists(vectorstore_dir):
    shutil.unpack_archive(zip_path, vectorstore_dir)
    print(f"✅ Unzipped vector store to: {vectorstore_dir}")
else:
    print(f"ℹ️ Directory already exists: {vectorstore_dir}")

# Load the vector store
vectorstore = Chroma(
    persist_directory=vectorstore_dir,
    embedding_function=embeddings_model
)

ℹ️ Directory already exists: ./pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb


# RAG
---

In [20]:
# OLlama download
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [21]:
# Launch the Ollama server locally
!ollama serve > /dev/null 2>&1 &
!sleep 10

In [22]:
!ollama pull llama2:latest
!pip install -U langchain-ollama

In [23]:
from langchain_ollama import OllamaLLM

In [24]:
llm = OllamaLLM(
    model="llama2:latest",
    temperature=0.05,
    system=""
)

## Personalized prompt:

In [28]:
prompt = ChatPromptTemplate.from_template(
"""
Medical Flashcard Generator Prompt

You will receive the name of a medical condition or disease. Your task is to create 5 comprehensive flashcards that systematically cover the essential aspects of the condition for medical education purposes.

Required Coverage Areas:
1. Definition & Pathophysiology - Core concept and underlying mechanisms
2. Etiology & Risk Factors - Causes and predisposing factors
3. Clinical Presentation - Signs, symptoms, and clinical manifestations
4. Diagnostic Approach - Key tests, criteria, and differential considerations
5. Management & Treatment - Therapeutic interventions and prognosis

Flashcard Requirements:
- Each flashcard must contain one focused question and one comprehensive answer
- Questions should be clinically relevant and test practical knowledge
- Answers should be precise, direct, and medically accurate
- Provide specific details (lab values, medication dosages, timeframes where applicable)
- Use medical terminology appropriately while maintaining clarity
- Give concise, focused responses without bullet points or lists
- Prioritize high-yield information commonly tested in medical examinations

Context Considerations:
{context}

Output Format:
Flashcard 1: [Topic Area]
Q: [Specific, focused question]
A: [Precise, direct answer in paragraph form]

Flashcard 2: [Topic Area]
Q: [Specific, focused question]
A: [Precise, direct answer in paragraph form]

[Continue for all 5 flashcards]

Quality Standards:
- Ensure medical accuracy and evidence-based content
- Use current clinical guidelines and best practices
- Include relevant mnemonics or memory aids where helpful
- Maintain consistency in terminology and formatting
- Focus on clinically actionable information

Medical Condition: {input}
"""
 )

## RAG Pipeline:

In [29]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# RAG Chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

## Inference Test:

### __Diabetes:__

In [30]:
question = "Diabetes"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1: Definition & Pathophysiology of Diabetes
Q: What is diabetes mellitus?
A: Diabetes mellitus is a group of metabolic disorders characterized by hyperglycemia (elevated blood glucose levels) resulting from defects in insulin secretion, insulin action, or both. The chronic hyperglycemia and attendant metabolic abnormalities of diabetes are often associated with secondary damage in multiple organ systems, especially the kidneys, eyes, nerves, and blood vessels.

Flashcard 2: Etiology & Risk Factors of Diabetes
Q: What causes type 1 diabetes?
A: Type 1 diabetes is an autoimmune disease in which the immune system mistakenly attacks and destroys the insulin-producing beta cells in the pancreas, leading to a complete deficiency of insulin production. The exact cause of this autoimmune response is still unknown, but it involves both genetic and environmental factors.

Flashcard 3: Clinical Presentation of Diabetes
Q: What are the clinical manifestations of diabetes?
A: T

### __Asthma:__

In [ ]:
question = "Asthma"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

### __Cardiac Arrest:__

In [ ]:
question = "Cardiac Arrest"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

### __Gastritis:__

In [ ]:
question = "Gastritis"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

### __Stroke:__

In [ ]:
question = "Stroke"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")